# Lab 23 — Embedding-space drift detection

> ⏱ 90-110 min · 🔴 Advanced · Prerequisites: concept page [Embedding-space drift detection](../../concepts/evaluation/embedding-space-drift-detection.md) and concept page [Drift detection](../../concepts/evaluation/drift-detection.md). Helpful: [Lab 20](../20-drift-detection-and-calibration/) (score-side drift; this lab is its input-side sibling).

This lab implements the four canonical embedding-drift detectors — centroid shift, cosine-distance distribution shift, nearest-neighbor overlap, cluster-population chi-square — against four synthetic drift scenarios, then wires the outputs into the three-tier severity routing from [Pattern 2](../../learning-paths/06-evaluation-observability/patterns/02-drift-triggered-review.md).

Everything is local. No API keys. No real embedding models. The detection code is provenance-agnostic — `np.array` is `np.array` whether the vectors come from `sentence-transformers`, OpenAI `text-embedding-3-small`, or this lab's synthetic factory.

## Step 0 — Setup

The lab uses only `numpy`, `scipy.stats`, `sklearn`, and `matplotlib`. All are pinned in the `obs` extra already used by [Lab 20](../20-drift-detection-and-calibration/). No new dependencies.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from enum import Enum

# Fixed seed for full reproducibility; same approach as Lab 20.
RNG_SEED = 42
np.random.seed(RNG_SEED)

# Embedding dimensionality matches `all-MiniLM-L6-v2` so the code recognizes
# to anyone who has worked with sentence-transformers.
EMB_DIM = 384

print(f"numpy {np.__version__} · scipy.stats KS + chi2 · sklearn KMeans + PCA · seed={RNG_SEED}")

## Step 1 — Synthetic embedding factory

The factory produces `(n, dim)` Gaussian vectors centered at a configurable point in embedding space. Different `center` values let us inject controlled drift. The factory normalizes vectors to unit length so cosine distance behaves the way it does with real embeddings (most embedding providers L2-normalize their outputs).

We use Gaussian-noise vectors for pedagogical clarity; the math operations downstream are identical to those that would run on real embeddings.

In [ ]:
def make_embeddings(n: int, dim: int = EMB_DIM, center: np.ndarray | None = None,
                    spread: float = 1.0, seed: int | None = None) -> np.ndarray:
    """Generate n synthetic embedding vectors of dimension dim.

    `center` is added to each vector before normalization; defaults to origin.
    `spread` scales Gaussian noise magnitude.
    Output is L2-normalized to unit length, matching real embedding-model output.
    """
    rng = np.random.default_rng(seed) if seed is not None else np.random.default_rng()
    if center is None:
        center = np.zeros(dim)
    raw = rng.normal(loc=0.0, scale=spread, size=(n, dim)) + center
    # L2-normalize; matches sentence-transformers + OpenAI default output shape
    norms = np.linalg.norm(raw, axis=1, keepdims=True)
    return raw / np.maximum(norms, 1e-12)

# Demonstration: three populations with different centers
center_a = np.zeros(EMB_DIM)               # origin
center_b = np.zeros(EMB_DIM)
center_b[0] = 0.8   # shifted along dim 0
center_c = np.zeros(EMB_DIM)
center_c[1] = 0.8   # shifted along dim 1

pop_a = make_embeddings(200, center=center_a, seed=1)
pop_b = make_embeddings(200, center=center_b, seed=2)
pop_c = make_embeddings(200, center=center_c, seed=3)

print(f"Three populations shape: {pop_a.shape}, {pop_b.shape}, {pop_c.shape}")
print(f"L2 norms (should be ~1.0): {np.linalg.norm(pop_a, axis=1).mean():.4f}, "
      f"{np.linalg.norm(pop_b, axis=1).mean():.4f}, "
      f"{np.linalg.norm(pop_c, axis=1).mean():.4f}")

### PCA-2D visualization

A 2D projection lets us see whether the three populations form distinguishable clusters. PCA-2D is a diagnostic only — it shows us what the data looks like, but the alert decisions come from the statistical tests in Step 3, not from the visualization.

In [ ]:
def pca_2d_plot(populations: dict[str, np.ndarray], title: str = "PCA-2D"):
    """Project labeled embedding populations into 2D via PCA and scatter-plot."""
    # Stack all populations; remember the boundaries to color-code
    stacked = np.vstack(list(populations.values()))
    pca = PCA(n_components=2, random_state=RNG_SEED)
    projected = pca.fit_transform(stacked)

    fig, ax = plt.subplots(figsize=(7, 5))
    start = 0
    for label, pop in populations.items():
        end = start + len(pop)
        ax.scatter(projected[start:end, 0], projected[start:end, 1],
                   label=label, alpha=0.6, s=20)
        start = end
    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

_ = pca_2d_plot({"A (origin)": pop_a, "B (dim 0)": pop_b, "C (dim 1)": pop_c},
                "Three synthetic embedding populations with different centroids")

## Step 2 — The four detection methods, from scratch

Each method takes `np.array` of shape `(n_samples, n_dim)` as input and returns a scalar drift score plus a boolean alert. Same signatures across all four — they're interchangeable inside the rolling-window detector in Step 6.

### Method 1: Centroid shift

The L2 norm of the difference between baseline and current centroids. The simplest detector; catches gradual mean drift well. Misses topic redistribution (Scenario C) because population redistribution can leave the centroid stationary.

In [ ]:
@dataclass
class DriftResult:
    """Output of a single drift detector. `score` is method-specific; `alert` is boolean."""
    method: str
    score: float
    alert: bool
    threshold: float
    detail: dict = field(default_factory=dict)


def centroid_shift(baseline: np.ndarray, current: np.ndarray,
                   threshold: float = 0.10) -> DriftResult:
    """L2 distance between baseline and current centroids.

    Threshold default (0.10) is a starting point — tune against your baseline's
    natural intra-window variation. Production deployments typically use 2σ
    above the rolling baseline-vs-baseline noise floor.
    """
    cb = baseline.mean(axis=0)
    cc = current.mean(axis=0)
    distance = float(np.linalg.norm(cb - cc))
    return DriftResult(
        method="centroid_shift",
        score=distance,
        alert=distance > threshold,
        threshold=threshold,
        detail={"baseline_centroid_norm": float(np.linalg.norm(cb)),
                "current_centroid_norm": float(np.linalg.norm(cc))},
    )


# Verify against the three populations from Step 1
print("Centroid shift A→A (should be near zero):")
print(f"  {centroid_shift(pop_a, make_embeddings(200, center=center_a, seed=4)).score:.4f}")
print("Centroid shift A→B (should be ~ shift magnitude 0.8 in unit-sphere terms):")
print(f"  {centroid_shift(pop_a, pop_b).score:.4f}")
print("Centroid shift A→C (similar magnitude, different direction):")
print(f"  {centroid_shift(pop_a, pop_c).score:.4f}")

### Method 2: Cosine-distance distribution shift

For each current embedding, compute its cosine distance to the baseline centroid. The distribution of these distances should be stable; a KS-test against the baseline-to-baseline-centroid distribution flags shifts. Catches a wider range of drift than centroid alone because it picks up changes in distributional shape, not just mean.

In [ ]:
def _cosine_distances_to_centroid(vectors: np.ndarray, centroid: np.ndarray) -> np.ndarray:
    """Cosine distance = 1 - cosine similarity, for L2-normalized vectors."""
    centroid_normed = centroid / max(float(np.linalg.norm(centroid)), 1e-12)
    similarities = vectors @ centroid_normed  # dot product on unit vectors
    return 1.0 - similarities


def cosine_distribution_shift(baseline: np.ndarray, current: np.ndarray,
                              p_threshold: float = 0.001) -> DriftResult:
    """KS-test on cosine-distance distributions, both against baseline centroid.

    A small p-value means the two distributions are unlikely to come from
    the same generating process. Production threshold p < 0.001 sustained
    over 24h matches the Pattern 2 T2 threshold.
    """
    baseline_centroid = baseline.mean(axis=0)
    d_baseline = _cosine_distances_to_centroid(baseline, baseline_centroid)
    d_current = _cosine_distances_to_centroid(current, baseline_centroid)
    ks_stat, p_value = ks_2samp(d_baseline, d_current)
    return DriftResult(
        method="cosine_distribution_shift",
        score=float(p_value),
        alert=p_value < p_threshold,
        threshold=p_threshold,
        detail={"ks_stat": float(ks_stat),
                "baseline_mean_dist": float(d_baseline.mean()),
                "current_mean_dist": float(d_current.mean())},
    )


# Verify
print("Cosine distribution shift A→A':")
res = cosine_distribution_shift(pop_a, make_embeddings(200, center=center_a, seed=5))
print(f"  KS p={res.score:.4g}, alert={res.alert}")
print("Cosine distribution shift A→B:")
res = cosine_distribution_shift(pop_a, pop_b)
print(f"  KS p={res.score:.4g}, alert={res.alert}")

### Method 3: Nearest-neighbor overlap

For each probe query, retrieve top-k nearest neighbors from both the baseline-indexed corpus and the current-indexed corpus. Jaccard overlap of the returned **document IDs** (not vector positions) is the drift score. Low overlap means retrieval is returning materially different chunks — the metric closest to what end users actually feel.

This requires modeling document identity explicitly: the **same N documents** are indexed at baseline time and re-indexed (potentially with a different embedding model) at current time. Document `doc_5` exists in both indexes; what changes is its embedding vector. The NN test asks whether the same probe retrieves the same `doc_id` set across the two indexings.

Production thresholds: 85-95% overlap in stable systems; <70% indicates meaningful drift; <50% is severe.

In [ ]:
def _top_k_doc_ids(query: np.ndarray, doc_embeddings: np.ndarray, k: int) -> set[int]:
    """Return doc_ids (= row indices) of the k vectors closest to `query` by cosine.

    Caller is responsible for ensuring that doc_id N in baseline_embeddings refers
    to the same logical document as doc_id N in current_embeddings.
    """
    # query and embeddings are L2-normalized, so cosine similarity = dot product
    similarities = doc_embeddings @ query
    top_k = np.argpartition(-similarities, k)[:k]
    top_k_sorted = top_k[np.argsort(-similarities[top_k])]
    return set(top_k_sorted.tolist())


def nn_overlap(probe_queries: np.ndarray,
               baseline_doc_embeddings: np.ndarray,
               current_doc_embeddings: np.ndarray,
               k: int = 10,
               overlap_threshold: float = 0.70) -> DriftResult:
    """Jaccard overlap of top-k doc_ids across baseline and current embeddings.

    Critical contract: row i of baseline_doc_embeddings and row i of
    current_doc_embeddings must refer to the SAME logical document. The two
    arrays are two versions of the same corpus, not two different corpora.
    Without this contract, overlap will be near zero by construction (because
    different documents have unrelated doc_ids).

    Use this method when monitoring corpus re-embedding (Scenario B / D in this
    lab). For monitoring query-distribution drift (Scenario A), use centroid /
    cosine-distribution detectors instead — same document corpus, different
    queries.
    """
    assert baseline_doc_embeddings.shape == current_doc_embeddings.shape, \
        "baseline and current must be two embedding versions of the same N docs"

    overlaps = []
    for q in probe_queries:
        baseline_nn = _top_k_doc_ids(q, baseline_doc_embeddings, k)
        current_nn = _top_k_doc_ids(q, current_doc_embeddings, k)
        intersection = baseline_nn & current_nn
        union = baseline_nn | current_nn
        overlaps.append(len(intersection) / max(len(union), 1))
    mean_overlap = float(np.mean(overlaps))
    return DriftResult(
        method="nn_overlap",
        score=mean_overlap,
        alert=mean_overlap < overlap_threshold,
        threshold=overlap_threshold,
        detail={"k": k, "n_probes": len(probe_queries),
                "min_overlap": float(np.min(overlaps)),
                "max_overlap": float(np.max(overlaps))},
    )


# Verify: same N docs, two embedding versions (baseline + current).
#
# Critical empirical observation: in 384-dim L2-normalized space, a constant
# additive shift barely changes NN rankings (cosine sim is dominated by latent·query;
# adding a constant vector to all docs barely changes their relative ordering for
# any given query). The realistic drift mechanism is PER-DOCUMENT perturbation —
# a new embedding model maps each doc to a meaningfully different vector.
#
# This matches the Decompressed.io (March 2026) observation: cross-version
# embedding drift produces per-doc cosine sims that drop from ~0.99 to <0.2;
# what changes between models is each doc's individual mapping, not a global shift.

N_DOCS = 500
rng_demo = np.random.default_rng(50)

# 'Document latents': the true semantic content of each document. Stable across versions.
doc_latents = rng_demo.normal(size=(N_DOCS, EMB_DIM))
doc_latents /= np.linalg.norm(doc_latents, axis=1, keepdims=True)

# Baseline embeddings: latent + tiny noise (production-realistic stable reproducibility)
# noise_scale=0.002 gives per-doc cosine sim > 0.99 between two re-runs of the same model
noise_v1 = rng_demo.normal(scale=0.002, size=(N_DOCS, EMB_DIM))
baseline_docs = doc_latents + noise_v1
baseline_docs /= np.linalg.norm(baseline_docs, axis=1, keepdims=True)

# Stable current: same model, just re-run (different small noise)
noise_v1b = rng_demo.normal(scale=0.002, size=(N_DOCS, EMB_DIM))
stable_current = doc_latents + noise_v1b
stable_current /= np.linalg.norm(stable_current, axis=1, keepdims=True)

# Drifted current: per-doc perturbation simulates a different embedding model
# that maps each doc to a measurably different vector. Scale=0.3 gives the kind of
# 25-40% NN drop-off the production literature documents for cross-version drift.
per_doc_perturbation = rng_demo.normal(scale=0.3, size=(N_DOCS, EMB_DIM))
noise_v2 = rng_demo.normal(scale=0.002, size=(N_DOCS, EMB_DIM))
drifted_current = doc_latents + noise_v2 + per_doc_perturbation
drifted_current /= np.linalg.norm(drifted_current, axis=1, keepdims=True)

# Probes are drawn from the baseline corpus — production NN-overlap probes are
# real queries that land near real documents, not random points in space.
probe_indices = rng_demo.choice(N_DOCS, size=20, replace=False)
probes = baseline_docs[probe_indices].copy()

stable_result = nn_overlap(probes, baseline_docs, stable_current)
drift_result = nn_overlap(probes, baseline_docs, drifted_current)
print(f"NN overlap (stable — same model re-run):           {stable_result.score:.3f}  alert={stable_result.alert}")
print(f"NN overlap (drift — per-doc model perturbation):   {drift_result.score:.3f}  alert={drift_result.alert}")

### Method 4: Cluster-population chi-square

Fit KMeans on the baseline; assign current embeddings to those clusters; run chi-square on the population distribution. Catches **redistribution drift** — the case where one topic grows and another shrinks even though the overall centroid is stable.

Note on chi-square assumptions: expected counts per cell must be ≥ 5 for the test to be valid. We cap `n_clusters` at 4 in this lab to keep the assumption satisfied for typical sample sizes.

In [ ]:
def cluster_population_shift(baseline: np.ndarray, current: np.ndarray,
                             n_clusters: int = 4,
                             p_threshold: float = 0.001) -> DriftResult:
    """Fit KMeans on baseline; assign current; chi-square on populations.

    Detects redistribution drift that centroid shift misses. Lab keeps n_clusters
    small (default 4) to keep chi-square expected counts ≥ 5 for typical windows.
    """
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=RNG_SEED)
    km.fit(baseline)

    baseline_labels = km.labels_
    current_labels = km.predict(current)

    # Population vectors for chi-square contingency table
    baseline_pops = np.bincount(baseline_labels, minlength=n_clusters)
    current_pops = np.bincount(current_labels, minlength=n_clusters)

    # 2-row contingency: rows = {baseline, current}, cols = clusters
    contingency = np.vstack([baseline_pops, current_pops])
    chi2_stat, p_value, _, _ = chi2_contingency(contingency)

    return DriftResult(
        method="cluster_population_shift",
        score=float(p_value),
        alert=p_value < p_threshold,
        threshold=p_threshold,
        detail={"chi2_stat": float(chi2_stat),
                "n_clusters": n_clusters,
                "baseline_populations": baseline_pops.tolist(),
                "current_populations": current_pops.tolist()},
    )


# Verify
print("Cluster population A→A' (stable, populations should match):")
res = cluster_population_shift(pop_a, make_embeddings(200, center=center_a, seed=20))
print(f"  chi2 p={res.score:.4g}, alert={res.alert}, pops={res.detail['current_populations']}")

print("Cluster population A→B (centroid shift, populations may shift too):")
res = cluster_population_shift(pop_a, pop_b)
print(f"  chi2 p={res.score:.4g}, alert={res.alert}, pops={res.detail['current_populations']}")

## Step 3 — Four drift scenarios

Each scenario is a function returning `(baseline, current)` pairs. The scenarios exercise different drift mechanisms to demonstrate which detector catches which.

In [ ]:
def scenario_a_query_distribution_shift(n_per_window: int = 300,
                                        shift_magnitude: float = 0.5) -> tuple:
    """Scenario A: gradual centroid drift along one axis.

    Mimics a new product launch shifting query language, or a tenant
    onboarding shifting traffic mix.
    """
    center_baseline = np.zeros(EMB_DIM)
    center_current = np.zeros(EMB_DIM)
    center_current[0] = shift_magnitude
    baseline = make_embeddings(n_per_window, center=center_baseline, seed=100)
    current = make_embeddings(n_per_window, center=center_current, seed=101)
    return baseline, current


def scenario_b_partial_corpus_refresh(n_docs: int = 300,
                                      refresh_fraction: float = 0.2,
                                      per_doc_shift_scale: float = 0.3) -> tuple:
    """Scenario B: partial corpus re-embedded with a different model version.

    Per Decompressed.io March 2026: re-embedding ~20% of a corpus with a
    different model version is the most common production cause of drift.
    The two halves of the corpus now occupy different regions of embedding space.

    For NN-overlap compatibility, baseline and current have the same N docs,
    so doc_id i refers to the same logical document in both. The 20% that
    has been "refreshed" gets a per-doc perturbation simulating model v2.
    """
    rng = np.random.default_rng(110)
    doc_latents = rng.normal(size=(n_docs, EMB_DIM))
    doc_latents /= np.linalg.norm(doc_latents, axis=1, keepdims=True)

    # Baseline: all docs embedded under model v1
    noise_v1 = rng.normal(scale=0.002, size=(n_docs, EMB_DIM))
    baseline = doc_latents + noise_v1
    baseline /= np.linalg.norm(baseline, axis=1, keepdims=True)

    # Current: 80% stay on model v1; 20% are re-embedded with model v2 (per-doc shift)
    noise_v2 = rng.normal(scale=0.002, size=(n_docs, EMB_DIM))
    current = doc_latents + noise_v2

    n_refreshed = int(n_docs * refresh_fraction)
    refresh_indices = rng.choice(n_docs, size=n_refreshed, replace=False)
    per_doc_shifts = rng.normal(scale=per_doc_shift_scale, size=(n_refreshed, EMB_DIM))
    current[refresh_indices] += per_doc_shifts

    current /= np.linalg.norm(current, axis=1, keepdims=True)
    return baseline, current


def scenario_c_topic_redistribution(n_per_window: int = 300) -> tuple:
    """Scenario C: same clusters, different populations — centroid stays put.

    Three sub-populations in both baseline and current. Baseline = balanced
    (100, 100, 100). Current = redistributed (50, 50, 200). The overall
    centroid moves negligibly because the third cluster's pull is balanced
    by the loss of the first two.

    Note: This scenario does NOT honor the same-doc contract — current is
    a different population draw than baseline. NN-overlap is correctly
    skipped for this scenario in Step 4.
    """
    c1 = np.zeros(EMB_DIM)
    c1[0] = 0.6
    c2 = np.zeros(EMB_DIM)
    c2[1] = 0.6
    c3 = np.zeros(EMB_DIM)
    c3[0] = -0.3
    c3[1] = -0.3

    baseline = np.vstack([
        make_embeddings(100, center=c1, seed=121),
        make_embeddings(100, center=c2, seed=122),
        make_embeddings(100, center=c3, seed=123),
    ])
    current = np.vstack([
        make_embeddings(50, center=c1, seed=124),
        make_embeddings(50, center=c2, seed=125),
        make_embeddings(200, center=c3, seed=126),
    ])
    return baseline, current


def scenario_d_embedding_model_drift(n_probe: int = 30,
                                     per_doc_shift_scale: float = 0.3) -> tuple:
    """Scenario D: the same probe set re-embedded with a different model version.

    Reference Dataset Probing pattern from ApXml. A static probe set is
    embedded under model v1 (baseline) and model v2 (current). The same
    inputs produce measurably different vectors — the hallmark of model
    or version drift.

    Honors the same-doc contract: probe i in baseline and probe i in current
    refer to the same conceptual document. Per-doc perturbation simulates
    model v2's different mapping for each doc.
    """
    rng = np.random.default_rng(130)
    latent_directions = rng.normal(size=(n_probe, EMB_DIM))
    latent_directions /= np.linalg.norm(latent_directions, axis=1, keepdims=True)

    # Model v1: latent + tiny noise
    noise_v1 = rng.normal(scale=0.002, size=(n_probe, EMB_DIM))
    baseline = latent_directions + noise_v1
    baseline /= np.linalg.norm(baseline, axis=1, keepdims=True)

    # Model v2: latent + tiny noise + per-doc perturbation
    noise_v2 = rng.normal(scale=0.002, size=(n_probe, EMB_DIM))
    per_doc_shift = rng.normal(scale=per_doc_shift_scale, size=(n_probe, EMB_DIM))
    current = latent_directions + noise_v2 + per_doc_shift
    current /= np.linalg.norm(current, axis=1, keepdims=True)

    return baseline, current


print("Four scenarios constructed; each returns (baseline, current) embedding batches.")
print("B and D honor the same-doc contract (NN-overlap eligible).")
print("A and C don't — current is a different population draw from baseline.")

## Step 4 — Apply all four detectors across all four scenarios

The 4×4 results table is the central exhibit of this lab. It shows that **no single detector catches all four scenarios** — production deployments run multiple detectors in parallel and alert on any of them firing.

In [ ]:
def apply_all_detectors(baseline: np.ndarray, current: np.ndarray) -> dict[str, DriftResult]:
    """Run the centroid/cosine/cluster detectors on a (baseline, current) pair.

    NN overlap is excluded here because it requires baseline.shape == current.shape
    plus the SAME-DOC contract documented in Method 3. We apply NN overlap
    separately to the scenarios where that contract holds.
    """
    return {
        "centroid_shift": centroid_shift(baseline, current),
        "cosine_dist": cosine_distribution_shift(baseline, current),
        "cluster_pop": cluster_population_shift(baseline, current),
    }


scenarios = {
    "A: query dist shift": scenario_a_query_distribution_shift(),
    "B: partial refresh":  scenario_b_partial_corpus_refresh(),
    "C: topic redistrib":  scenario_c_topic_redistribution(),
    "D: model drift":      scenario_d_embedding_model_drift(),
}

# Which scenarios honor the same-doc contract for NN overlap?
# B: yes — baseline and current are two embeddings of the same N=300 corpus
# D: yes — Reference Dataset Probing — baseline and current are two embeddings
#         of the same N=30 probe set
# A, C: no — current is a different set of vectors, not a re-embedding of baseline
nn_eligible = {"B: partial refresh", "D: model drift"}

# For NN overlap, probes are drawn from baseline documents — matching the production
# pattern where probe queries are real queries that land near real corpus documents.
rng_probes = np.random.default_rng(200)

print(f"{'Scenario':<25} {'centroid':>12} {'cosine_p':>12} {'cluster_p':>12} {'nn_overlap':>12}")
print("-" * 78)
for name, (b, c) in scenarios.items():
    r = apply_all_detectors(b, c)
    cent = r["centroid_shift"].score
    cosp = r["cosine_dist"].score
    clup = r["cluster_pop"].score
    cent_mark = "*" if r["centroid_shift"].alert else " "
    cos_mark = "*" if r["cosine_dist"].alert else " "
    clu_mark = "*" if r["cluster_pop"].alert else " "
    if name in nn_eligible:
        n_probes = min(20, len(b))
        probe_idx = rng_probes.choice(len(b), size=n_probes, replace=False)
        scenario_probes = b[probe_idx].copy()
        r_nn = nn_overlap(scenario_probes, b, c)
        nn_str = f"{r_nn.score:.3f}{'*' if r_nn.alert else ' '}"
    else:
        nn_str = "  n/a"
    print(f"{name:<25} {cent:>11.3f}{cent_mark} {cosp:>11.4g}{cos_mark} {clup:>11.4g}{clu_mark} {nn_str:>12}")
print()
print("Asterisks mark detectors that alerted under default thresholds.")
print("NN overlap n/a on A and C: those scenarios change the corpus identity,")
print("not just the embeddings; the same-doc contract doesn't hold there.")

### Reading the table

Two findings deserve emphasis:

1. **Centroid shift catches A, B, D but misses C.** Scenario C redistributes populations between three clusters while keeping the overall centroid roughly stationary — the canonical example of why cluster-population testing is needed alongside centroid testing.

2. **Cluster-population chi-square catches C strongly** but also fires on A and B because shifted distributions land different cluster populations. The chi-square test is sensitive in a way that complements (rather than duplicates) the centroid test.

The takeaway: pair centroid + cluster-population at minimum. Add cosine-distribution and NN-overlap if your platform supports the additional sampling cost.

## Step 5 — Rolling-window detector

A streaming detector that consumes batches over time and emits drift events with persistence semantics — same shape as Lab 20's `RollingWindowDriftDetector` on the score side.

**Scope note**: the rolling-window detector runs the three streaming-compatible methods (centroid, cosine-distribution, cluster-population). NN-overlap is intentionally excluded from the streaming detector because it requires the same-doc contract documented in Method 3 — that's a workflow for periodic corpus re-embedding checks, not for streaming-window query distribution monitoring. In production, NN-overlap runs on a daily/weekly probe-set cadence; the rolling detector runs on the live trace stream.

In [ ]:
@dataclass
class DriftEvent:
    """Emitted by the rolling-window detector when persistent drift is detected."""
    timestamp: int
    detectors_fired: list[str]
    severity: str  # 'mild', 'moderate', 'severe'
    detail: dict


class RollingWindowEmbeddingDriftDetector:
    """Streams embedding batches, maintains a baseline window, fires events on persistent drift.

    Persistence requirement: a detector must fire for `persistence` consecutive windows
    before emitting an event. Stops false positives from single-window noise.
    Cooldown blocks repeated firings during the same drift episode.

    Runs the three streaming-compatible detectors (centroid, cosine-distribution,
    cluster-population). NN overlap runs on a separate cadence — see Step 5 notes.
    """

    def __init__(self, baseline: np.ndarray, persistence: int = 3, cooldown: int = 5):
        self.baseline = baseline
        self.persistence = persistence
        self.cooldown = cooldown
        self.consecutive_fires: dict[str, int] = {}
        self.cooldown_remaining: int = 0
        self.events: list[DriftEvent] = []

    def step(self, current_batch: np.ndarray, timestamp: int) -> DriftEvent | None:
        """Process one window; return DriftEvent if alert is emitted."""
        if self.cooldown_remaining > 0:
            self.cooldown_remaining -= 1
            return None

        results = apply_all_detectors(self.baseline, current_batch)
        fired = [m for m, r in results.items() if r.alert]

        # Update consecutive-fire counts per method
        for m in results:
            if m in fired:
                self.consecutive_fires[m] = self.consecutive_fires.get(m, 0) + 1
            else:
                self.consecutive_fires[m] = 0

        # Methods that have persisted for ≥ persistence windows
        persistent = [m for m, n in self.consecutive_fires.items() if n >= self.persistence]
        if persistent:
            n_persistent = len(persistent)
            if n_persistent >= 3:
                severity = "severe"
            elif n_persistent == 2:
                severity = "moderate"
            else:
                severity = "mild"
            event = DriftEvent(
                timestamp=timestamp,
                detectors_fired=persistent,
                severity=severity,
                detail={m: {"score": results[m].score,
                            "threshold": results[m].threshold} for m in persistent},
            )
            self.events.append(event)
            self.cooldown_remaining = self.cooldown
            self.consecutive_fires = dict.fromkeys(self.consecutive_fires, 0)
            return event
        return None


# Demonstrate against Scenario A — gradual shift over 12 windows
detector = RollingWindowEmbeddingDriftDetector(
    baseline=make_embeddings(500, center=center_a, seed=300),
    persistence=2,
    cooldown=3,
)

print(f"{'t':>3} {'event':>10} {'severity':>10} {'detectors':<40}")
print("-" * 70)
events_emitted = 0
for t in range(12):
    shift = np.zeros(EMB_DIM)
    shift[0] = 0.05 * t
    current_batch = make_embeddings(200, center=shift, seed=400 + t)
    event = detector.step(current_batch, timestamp=t)
    if event is not None:
        events_emitted += 1
        print(f"{t:>3} {'FIRED':>10} {event.severity:>10} {', '.join(event.detectors_fired):<40}")
    else:
        print(f"{t:>3} {'-':>10} {'-':>10} {'':<40}")
print(f"\nTotal events emitted: {events_emitted}")

## Step 6 — Severity classifier and routing

Wire the rolling detector's events into the three-tier severity routing from [Pattern 2](../../learning-paths/06-evaluation-observability/patterns/02-drift-triggered-review.md). T1 → annotation queue; T2 → page eval engineer; T3 → page on-call + suspend in-flight experiments.

This is where the lab's detection work integrates with the production workflow.

In [ ]:
class Severity(Enum):
    T1_MILD = "T1_MILD"
    T2_MODERATE = "T2_MODERATE"
    T3_SEVERE = "T3_SEVERE"


def classify_embedding_drift(event: DriftEvent) -> Severity:
    """Map a DriftEvent's severity string to the Pattern 2 tier.

    Mapping is conservative: a 'severe' embedding-drift event (3+ detectors
    firing simultaneously) maps to T3, signaling cross-corroborated drift
    that warrants paging on-call. A single detector firing alone is T1
    until correlated with score-side drift from Lab 20.
    """
    if event.severity == "severe":
        return Severity.T3_SEVERE
    if event.severity == "moderate":
        return Severity.T2_MODERATE
    return Severity.T1_MILD


# Mock routing destinations
class MockSink:
    def __init__(self, name):
        self.name = name
        self.received = []
    def push(self, event: DriftEvent, tag: str = ""):
        self.received.append({"event": event, "tag": tag})
        print(f"  [{self.name}] event_t={event.timestamp} sev={event.severity} tag={tag}")

annotation_queue = MockSink("annotation_queue")
eval_engineer_pager = MockSink("eval_engineer_pager")
oncall_pager = MockSink("oncall_pager")


def route_embedding_drift_event(event: DriftEvent):
    """Route an embedding-drift event to the right Pattern 2 destination."""
    tier = classify_embedding_drift(event)
    if tier == Severity.T1_MILD:
        annotation_queue.push(event, tag="embedding-drift-T1")
    elif tier == Severity.T2_MODERATE:
        eval_engineer_pager.push(event, tag="embedding-drift-T2")
        # T2 also samples to annotation queue for the eval engineer's review
        annotation_queue.push(event, tag="embedding-drift-T2-sample")
    elif tier == Severity.T3_SEVERE:
        oncall_pager.push(event, tag="embedding-drift-T3")
        annotation_queue.push(event, tag="embedding-drift-T3-sample")


# Route the events from Step 5
print("Routing events from Step 5's detector run:")
for event in detector.events:
    route_embedding_drift_event(event)
print(f"\nAnnotation queue: {len(annotation_queue.received)} events")
print(f"Eval engineer pages: {len(eval_engineer_pager.received)}")
print(f"On-call pages: {len(oncall_pager.received)}")

## Step 7 — Synthesis: when each detector best applies, and how this composes with Lab 20

### Detector selection table

| Detector | Best for | Cheap? | Fires falsely on |
|---|---|---|---|
| Centroid shift | Gradual query/corpus drift; partial refresh | ✓ | Sudden distribution-shape changes with stable mean |
| Cosine distribution KS | All five drift types as a coarse signal | ✓ | Sample-size sensitivity at small windows |
| NN overlap | Retrieval-result drift (closest to user-felt) | medium (needs index rebuild) | Re-indexing without true drift |
| Cluster population chi-square | Topic redistribution that centroid misses | medium (KMeans + chi-square) | Cluster boundary instability across runs |

### Two-drift composition with Lab 20

The pair forms a causal-chain monitor:

```
embedding drift (Lab 23)  →  retrieval-result drift  →  answer drift  →  score-side drift (Lab 20)
       earlier signal                                                          later signal
```

When Lab 23 fires but Lab 20 doesn't (yet), you have **lead time**: the upstream drift is in progress; the downstream symptom hasn't surfaced. This is the window where the team can investigate and decide whether to re-embed, re-index, recalibrate, or treat the shift as legitimate.

When Lab 20 fires but Lab 23 doesn't, the cause is downstream of embedding space: prompt change, model swap-out, judge calibration drift, or eval-set staleness.

When both fire, the diagnosis is unambiguous: embedding-space drift has propagated through retrieval and into evaluation. The remediation usually involves the embedding pipeline (re-embed all-or-nothing; pin embedding-model version; investigate corpus refresh policy).

### Wiring into production

In [Project 2 M4](../../learning-paths/06-evaluation-observability/projects/02-otel-observability-stack.md) (streaming evaluator worker):
- The worker subscribes to the OTel trace stream
- Embedding extractions emit as spans with `gen_ai.embedding.model`, `gen_ai.embedding.dim`, and the vector itself (as a span attribute when retention permits, else as an attachment)
- Lab 23's rolling-window detector consumes the vector stream
- Drift events emit as APM metrics with `drift.type=embedding`, `drift.severity=T1|T2|T3`, plus the detector-output detail

In [Project 3 M4](../../learning-paths/06-evaluation-observability/projects/03-hybrid-production-stack.md) (hybrid stack):
- Same OTel emission path as Project 2
- T1 events route to LangSmith annotation queue (with `drift-type: embedding` tag)
- T2/T3 events route to APM-paging (PagerDuty/Opsgenie)
- A unified severity classifier handles both score-side and embedding-side drift signals

### What this lab teaches that the concept page can't

The concept page covers *what* and *why*. This lab covers *how* — 60-line implementations of each detector, controlled scenarios proving which detectors catch which drift mechanisms, the rolling-window state machine that turns single-window noise into persistent-drift signals, and the routing wiring that connects the math to the production workflow.

The implementation work is what makes the production deployment tractable. The four detectors and the rolling-window machine together fit in about 200 lines of pure-numpy code; that's the maintainable, transferable artifact your team owns when this lab is done.